In [0]:
%run /Users/arnaldo.jbj@gmail.com/ElectricalGeneratedDistribution-Project/pipelines/02_extracting_from_bronze_to_treatments_into_silver/step00_IMPORTS

In [0]:
%run /Users/arnaldo.jbj@gmail.com/ElectricalGeneratedDistribution-Project/pipelines/02_extracting_from_bronze_to_treatments_into_silver/step00_AUXILIARY_FUNCTIONS

In [0]:
spark.table('bronze.parametros_lei_14300').show(truncate=False)

In [0]:
TABLE_ORIGIN = 'parametros_lei_14300'

TABLE_DESTINY = 'law_14300_parameters'

df = (
        spark.table(f"bronze.{TABLE_ORIGIN}")
             .filter(F.col('ano') != 'corte_gd1')
             .withColumnsRenamed({
                                   'ano' : 'year_start_contract',
                                   'percentual_fio_b_nao_compensado' : 'pct_b_wire_not_compensated'
                                })
             .withColumn('pct_b_wire_not_compensated', F.col('pct_b_wire_not_compensated').cast('float'))
     )

# Incremental load: never replaces silver.law_14300_parameters — appends only new years
target = f"silver.{TABLE_DESTINY}"
key_cols = ['year_start_contract']

if not spark.catalog.tableExists(target):
    # first load: table does not exist yet — create it
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target)
    print(f"✓ {target} created ({df.count()} rows)")
else:
    # append only records whose key is not present in silver yet
    df_new = df.join(spark.table(target).select(*key_cols).distinct(), on=key_cols, how='left_anti')
    n_new = df_new.count()
    if n_new:
        df_new.write.format("delta").mode("append").saveAsTable(target)
        print(f"✓ {target} appended {n_new} new row(s)")
    else:
        print(f"↷ {target}: no new records — nothing appended")